# Modelling

Pipeline is as follows:
1. Create logistic regression
2. Create random forest and XGboost
3. Compare step 2 with step 1
4. Decide on a model to feature engineer and model for deployment

This pipline is designed to create the best possible model. Data should be split into train test, with cross validation at the end.

In [100]:
import pandas as pd 
import numpy as np 

data = pd.read_csv('data/pairwise_data.csv')
data.head()

,seed_track_id,similar_track_id,popularity_diff,danceability_diff,energy_diff,loudness_diff,speechiness_diff,acousticness_diff,instrumentalness_diff,liveness_diff,valence_diff,tempo_diff,duration_s_diff,same_genre,same_artist,same_album,target
0,0000vdREvCVMxbQTkS888c,2DX4Lj4wXV3VK1F8yl0x54,7.0,0.019,0.190,1.524,0.130,0.0317,0.00301,0.0420,0.059,8.018,2.725,1,0,0,1
1,0000vdREvCVMxbQTkS888c,4bYH5445Bn2w9UiGM0NxQw,4.0,0.029,0.092,1.063,0.027,0.0498,0.01159,0.0530,0.281,36.008,41.272,1,0,0,1
2,0000vdREvCVMxbQTkS888c,5u7QargOKXO82nXcAU9s8L,2.0,0.052,0.222,2.637,0.065,0.0077,0.00300,0.0480,0.273,23.962,57.692,1,0,0,1
3,0000vdREvCVMxbQTkS888c,0wi1ky0zqwG8195E4omWui,11.0,0.033,0.135,1.637,0.049,0.0102,0.00301,0.0735,0.147,7.975,23.565,0,0,0,1
4,0000vdREvCVMxbQTkS888c,6wKKcAUaLYDy7bN1uFD0E5,11.0,0.088,0.049,1.893,0.055,0.2923,0.00301,0.0668,0.046,10.991,23.264,0,0,0,1


In [101]:
#split into X and y
X = data.drop(columns = ['seed_track_id', 'similar_track_id', 'target'])
y = data['target']

In [102]:
#train test validaiton split
# from sklearn.model_selection import train_test_split   
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# this is bad as it could break the grouping of seed_track_id and similar_track_id, we want to make sure that all pairs of a given seed_track_id are in the same set (train, test, validation)
# we can do this by using the GroupShuffleSplit from sklearn


In [103]:
# group shuffle split
from sklearn.model_selection import GroupShuffleSplit   
split = GroupShuffleSplit(test_size=0.2, random_state=42)
train_idx, test_idx = next(split.split(X, y, groups=data["seed_track_id"]))

X_train = X.iloc[train_idx]
y_train = y.iloc[train_idx]

X_test = X.iloc[test_idx]
y_test = y.iloc[test_idx]

### Logistic Regresion

In [104]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import cross_validate, StratifiedKFold

log_model = LogisticRegression(max_iter=1000)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_results = cross_validate(
    log_model,
    X_train,
    y_train,
    cv=cv,
    scoring=["accuracy", "precision_macro", "recall_macro", "f1_macro"],
    return_train_score=False
)

cv_df = pd.DataFrame(cv_results)
print("Cross-validation results:")
print(cv_df)

print("\nAverage cross-validation scores:")
print(cv_df.mean())

log_model.fit(X_train, y_train)

log_pred = log_model.predict(X_test)

print("\nTest set classification report:")
print(classification_report(y_test, log_pred))

Cross-validation results:
   fit_time  score_time  test_accuracy  test_precision_macro  \
0  5.674811    0.023235       0.977588              0.977614   
1  5.639573    0.030322       0.977839              0.977881   
2  3.985147    0.022611       0.977289              0.977315   
3  5.281562    0.023009       0.977268              0.977298   
4  5.233170    0.022028       0.977818              0.977835   

   test_recall_macro  test_f1_macro  
0           0.977588       0.977588  
1           0.977839       0.977838  
2           0.977289       0.977288  
3           0.977268       0.977267  
4           0.977818       0.977818  

Average cross-validation scores:
fit_time                5.162853
score_time              0.024241
test_accuracy           0.977560
test_precision_macro    0.977589
test_recall_macro       0.977560
test_f1_macro           0.977560
dtype: float64

Test set classification report:
              precision    recall  f1-score   support

           0       0.98   

### Random Forest Model

In [105]:
from sklearn.ensemble import RandomForestClassifier


rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_results = cross_validate(
    rf_model,
    X_train,
    y_train,
    cv=cv,
    scoring=["accuracy", "precision_macro", "recall_macro", "f1_macro"],
    return_train_score=False
)

cv_df = pd.DataFrame(cv_results)

print("Cross-validation results:")
print(cv_df)

print("\nAverage cross-validation scores:")
print(cv_df.mean())


rf_model.fit(X_train, y_train)


rf_pred = rf_model.predict(X_test)

print("\nTest set classification report:")
print(classification_report(y_test, rf_pred))

Cross-validation results:
    fit_time  score_time  test_accuracy  test_precision_macro  \
0  34.507734    0.366101       0.982226              0.982229   
1  35.578270    0.389495       0.982547              0.982547   
2  36.369317    0.370296       0.982115              0.982115   
3  33.987659    0.357717       0.981941              0.981942   
4  34.449992    0.361651       0.982219              0.982222   

   test_recall_macro  test_f1_macro  
0           0.982226       0.982226  
1           0.982547       0.982547  
2           0.982115       0.982115  
3           0.981941       0.981941  
4           0.982219       0.982219  

Average cross-validation scores:
fit_time                34.978594
score_time               0.369052
test_accuracy            0.982210
test_precision_macro     0.982211
test_recall_macro        0.982210
test_f1_macro            0.982210
dtype: float64

Test set classification report:
              precision    recall  f1-score   support

           0  

### XG Boost

In [106]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    n_jobs=-1,
    eval_metric="logloss"
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_results = cross_validate(
    xgb_model,
    X_train,
    y_train,
    cv=cv,
    scoring=["accuracy", "precision_macro", "recall_macro", "f1_macro"],
    return_train_score=False
)

cv_df = pd.DataFrame(cv_results)

print("Cross-validation results:")
print(cv_df)

print("\nAverage cross-validation scores:")
print(cv_df.mean())

xgb_model.fit(X_train, y_train)

xgb_pred = xgb_model.predict(X_test)

print("\nTest set classification report:")
print(classification_report(y_test, xgb_pred))

Cross-validation results:
   fit_time  score_time  test_accuracy  test_precision_macro  \
0  1.490751    0.076926       0.981182              0.981188   
1  1.589889    0.080319       0.981669              0.981689   
2  1.599574    0.079563       0.981398              0.981409   
3  1.569066    0.080955       0.981001              0.981009   
4  1.605327    0.071918       0.981843              0.981848   

   test_recall_macro  test_f1_macro  
0           0.981182       0.981182  
1           0.981669       0.981669  
2           0.981398       0.981398  
3           0.981001       0.981001  
4           0.981843       0.981843  

Average cross-validation scores:
fit_time                1.570921
score_time              0.077936
test_accuracy           0.981419
test_precision_macro    0.981428
test_recall_macro       0.981419
test_f1_macro           0.981418
dtype: float64

Test set classification report:
              precision    recall  f1-score   support

           0       0.98   

In [107]:


xgb_report = classification_report(y_test, xgb_pred, output_dict=True)
rf_report = classification_report(y_test, rf_pred, output_dict=True)
log_report = classification_report(y_test, log_pred, output_dict=True)

comparison_table = pd.DataFrame([
    {
        "model": "XGBoost",
        "accuracy": xgb_report["accuracy"],
        "precision_macro": xgb_report["macro avg"]["precision"],
        "recall_macro": xgb_report["macro avg"]["recall"],
        "f1_macro": xgb_report["macro avg"]["f1-score"],
        "precision_weighted": xgb_report["weighted avg"]["precision"],
        "recall_weighted": xgb_report["weighted avg"]["recall"],
        "f1_weighted": xgb_report["weighted avg"]["f1-score"],
    },
    {
        "model": "Random Forest",
        "accuracy": rf_report["accuracy"],
        "precision_macro": rf_report["macro avg"]["precision"],
        "recall_macro": rf_report["macro avg"]["recall"],
        "f1_macro": rf_report["macro avg"]["f1-score"],
        "precision_weighted": rf_report["weighted avg"]["precision"],
        "recall_weighted": rf_report["weighted avg"]["recall"],
        "f1_weighted": rf_report["weighted avg"]["f1-score"],
    },
    {
        "model": "Logistic Regression",
        "accuracy": log_report["accuracy"],
        "precision_macro": log_report["macro avg"]["precision"],
        "recall_macro": log_report["macro avg"]["recall"],
        "f1_macro": log_report["macro avg"]["f1-score"],
        "precision_weighted": log_report["weighted avg"]["precision"],
        "recall_weighted": log_report["weighted avg"]["recall"],
        "f1_weighted": log_report["weighted avg"]["f1-score"],
    }
])

print(comparison_table)


                 model  accuracy  precision_macro  recall_macro  f1_macro  \
0              XGBoost  0.982199         0.982214      0.982199  0.982198   
1        Random Forest  0.983029         0.983029      0.983029  0.983029   
2  Logistic Regression  0.978649         0.978683      0.978649  0.978649   

   precision_weighted  recall_weighted  f1_weighted  
0            0.982214         0.982199     0.982198  
1            0.983029         0.983029     0.983029  
2            0.978683         0.978649     0.978649  


In [108]:
from sklearn.metrics import confusion_matrix

log_cm = confusion_matrix(y_test, log_pred)
rf_cm = confusion_matrix(y_test, rf_pred)
xgb_cm = confusion_matrix(y_test, xgb_pred)

print("Logistic Regression\n", log_cm)
print("Random Forest\n", rf_cm)
print("XGBoost\n", xgb_cm)

Logistic Regression
 [[87449  2291]
 [ 1541 88199]]
Random Forest
 [[88211  1529]
 [ 1517 88223]]
XGBoost
 [[87885  1855]
 [ 1340 88400]]


In [112]:
#comparing false positive rates
fp_rate_log = log_cm[0][1] / (log_cm[0][0] + log_cm[0][1])
fp_rate_rf = rf_cm[0][1] / (rf_cm[0][0] + rf_cm[0][1])
fp_rate_xgb = xgb_cm[0][1] / (xgb_cm[0][0] + xgb_cm[0][1]) 
print("\nFalse Positive Rates:")
print(f"Logistic Regression: {fp_rate_log:.4f}")
print(f"Random Forest: {fp_rate_rf:.4f}")
print(f"XGBoost: {fp_rate_xgb:.4f}")


False Positive Rates:
Logistic Regression: 0.0255
Random Forest: 0.0170
XGBoost: 0.0207


### Model Evaluation

Across accuracy ($\frac{TP + TN}{TP + FP + TN + FN}$), precision ($\frac{TP}{TP + FP}$), recall ($\frac{TP}{TP+FN}$) and F1 ($\frac{2 \times \text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$), random forrest performs the best. 

Looking at a business context, false positives may be more harmful as customers will be more likely to leave the streaming platform if they are given too many false positive reccomendations. Upon exploring false positive rates ($\frac{FP}{FP + TN}$), it was found that Random Forrests once again rank better in this area. 

However, the issue with Random Forrests is the time it takes to run the output; XGboost takes ~ 10 seconds, whereas Random Forrests takes ~3.5 minutes. Extra trees will also be explored as it is faster than random forrests as it uses more randomness for split points, rather than computing optimal performance each time. It lowers variances, however bias is increased. 

In [115]:
from sklearn.ensemble import ExtraTreesClassifier

et_model = ExtraTreesClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)


cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_results = cross_validate(
    et_model,
    X_train,
    y_train,
    cv=cv,
    scoring=["accuracy", "precision_macro", "recall_macro", "f1_macro"],
    return_train_score=False
)

cv_df = pd.DataFrame(cv_results)

print("Cross-validation results:")
print(cv_df)

print("\nAverage CV performance:")
print(cv_df.mean())
      
et_model.fit(X_train, y_train)
et_pred = et_model.predict(X_test)

print(classification_report(y_test, et_pred))

Cross-validation results:
   fit_time  score_time  test_accuracy  test_precision_macro  \
0  7.983050    0.550949       0.981899              0.981901   
1  7.780874    0.526000       0.982658              0.982664   
2  7.782278    0.532063       0.982289              0.982293   
3  8.103895    0.520510       0.981955              0.981958   
4  7.777943    0.684685       0.982150              0.982150   

   test_recall_macro  test_f1_macro  
0           0.981899       0.981899  
1           0.982658       0.982658  
2           0.982289       0.982289  
3           0.981955       0.981955  
4           0.982150       0.982150  

Average CV performance:
fit_time                7.885608
score_time              0.562841
test_accuracy           0.982190
test_precision_macro    0.982193
test_recall_macro       0.982190
test_f1_macro           0.982190
dtype: float64
              precision    recall  f1-score   support

           0       0.98      0.98      0.98     89740
           1  

In [121]:
#new comparison table with extra trees
et_report = classification_report(y_test, et_pred, output_dict=True)

comparison_table = pd.DataFrame([
    {
        "model": "XGBoost",
        "accuracy": xgb_report["accuracy"],
        "precision_macro": xgb_report["macro avg"]["precision"],
        "recall_macro": xgb_report["macro avg"]["recall"],
        "f1_macro": xgb_report["macro avg"]["f1-score"],
        "precision_weighted": xgb_report["weighted avg"]["precision"],
        "recall_weighted": xgb_report["weighted avg"]["recall"],
        "f1_weighted": xgb_report["weighted avg"]["f1-score"],
    },
    {
        "model": "Random Forest",
        "accuracy": rf_report["accuracy"],
        "precision_macro": rf_report["macro avg"]["precision"],
        "recall_macro": rf_report["macro avg"]["recall"],
        "f1_macro": rf_report["macro avg"]["f1-score"],
        "precision_weighted": rf_report["weighted avg"]["precision"],
        "recall_weighted": rf_report["weighted avg"]["recall"],
        "f1_weighted": rf_report["weighted avg"]["f1-score"],
    },  

    {
        "model": "Extra Trees",
        "accuracy": et_report["accuracy"],
        "precision_macro": et_report["macro avg"]["precision"],
        "recall_macro": et_report["macro avg"]["recall"],
        "f1_macro": et_report["macro avg"]["f1-score"],
        "precision_weighted": et_report["weighted avg"]["precision"],
        "recall_weighted": et_report["weighted avg"]["recall"],
        "f1_weighted": et_report["weighted avg"]["f1-score"],
    }, 
    {
        "model": "Logistic Regression",
        "accuracy": log_report["accuracy"],
        "precision_macro": log_report["macro avg"]["precision"],
        "recall_macro": log_report["macro avg"]["recall"],
        "f1_macro": log_report["macro avg"]["f1-score"],
        "precision_weighted": log_report["weighted avg"]["precision"],
        "recall_weighted": log_report["weighted avg"]["recall"],
        "f1_weighted": log_report["weighted avg"]["f1-score"],
    }
])

comparison_table  

,model,accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted
0,XGBoost,0.982199,0.982214,0.982199,0.982198,0.982214,0.982199,0.982198
1,Random Forest,0.983029,0.983029,0.983029,0.983029,0.983029,0.983029,0.983029
2,Extra Trees,0.982967,0.982971,0.982967,0.982967,0.982971,0.982967,0.982967
3,Logistic Regression,0.978649,0.978683,0.978649,0.978649,0.978683,0.978649,0.978649


In [118]:
# all confusion matrix values
et_cm = confusion_matrix(y_test, et_pred)

print("Logistic Regression\n", log_cm)
print("Random Forest\n", rf_cm)
print("XGBoost\n", xgb_cm)
print("Extra Trees\n", et_cm)


Logistic Regression
 [[87449  2291]
 [ 1541 88199]]
Random Forest
 [[88211  1529]
 [ 1517 88223]]
XGBoost
 [[87885  1855]
 [ 1340 88400]]
Extra Trees
 [[88083  1657]
 [ 1400 88340]]


In [122]:
# all false positive rate
fp_rate_et = et_cm[0][1] / (et_cm[0][0] + et_cm[0][1])

print("\nFalse Positive Rates:")
print(f"Logistic Regression: {fp_rate_log:.4f}")
print(f"Random Forest: {fp_rate_rf:.4f}")
print(f"XGBoost: {fp_rate_xgb:.4f}")
print(f"Extra Trees: {fp_rate_et:.4f}")


False Positive Rates:
Logistic Regression: 0.0255
Random Forest: 0.0170
XGBoost: 0.0207
Extra Trees: 0.0185


### Final Evaluation

Looking at all models it is clear that random forrests performs slightly better across all metrics. However, its long training time is impractical for a music streaming platform's function, where predictions need to be generated quickly. Therefore, XGboost will be chosen as the final model, and hyperperameter tuning will be used to try and beat the benchmark performance set by random forrests.